# VGGT Depth — Untargeted PGD Attack

Adversarial robustness probe of VGGT's depth head on a DTU scene. The attack searches for an L∞-bounded image perturbation that maximizes a **scale-invariant log-depth divergence** from the clean prediction, weighted by the clean depth confidence.

Why this loss vs. the paper's plain L1 regression term:
- The VGGT paper's depth loss is the aleatoric-uncertainty L1: $\mathcal{L}_{\text{depth}} = \sum_i \|\Sigma_i^D \odot (\hat{D}_i - D_i)\| + (\text{gradient term}) - \alpha \log \Sigma_i^D$. Maximising the bare $|D_{\text{adv}} - D_{\text{clean}}|$ as an attack rewards a **uniform** multiplicative shift of every pixel's depth — PGD finds that direction in a few steps, but the evaluator's ICP-with-scale Umeyama alignment absorbs uniform scale and the Chamfer barely moves. We confirmed this empirically: matched conf threshold of 1.001 gave Overall ≈ 0.91 mm for the L1 attack (no worse than clean) despite a much larger raw-loss number.
- VGGT depth uses an `exp` activation, so $D > 0$ everywhere. The Eigen-style scale-invariant log loss (the *variance* of $\log D_{\text{adv}} - \log D_{\text{clean}}$, weighted by clean confidence) subtracts off the global mean by construction, so a uniform rescale of every pixel gives **zero** loss. The attacker is forced to distort *relative* geometry — exactly what the Chamfer metric measures after alignment.
- Confidence weighting (using clean `depth_conf`, detached) concentrates the attack where the model is most certain. Those are the predictions that downstream fusion / SfM actually trust, and freezing the weight at clean values closes the "lower my own confidence to hide" escape hatch that the point-map attack also had to plug.

In short: SI-log is the same paper-spirited "match training-loss form" idea, with one modification that closes the global-scale escape hatch the L1 form creates for depth specifically. Outputs are written to `output/` and `images/` for an evaluation script to consume.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.utils as vutils

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # The old notebook set deterministic=True AND benchmark=True simultaneously,
    # which is contradictory; benchmark picks the fastest kernel per-shape and
    # defeats determinism. Pick one. We pick determinism for a security probe.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 0
seed_everything(SEED)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Use the supported HuggingFace loader (PyTorchModelHubMixin) instead of
# torch.hub.load_state_dict_from_url. This gives us the cached path and avoids
# the implicit pickle-load behavior of the old loader.
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# We run the attack in fp32 for stable gradients. VGGT's prediction heads
# already force fp32 internally via `with torch.cuda.amp.autocast(enabled=False)`,
# and the aggregator is fine in fp32 since gradient flow into delta is what we
# care about, not aggregator throughput.

In [ ]:
DTU_SCAN = Path("/u/nleobandung/dtu/SampleSet/MVS Data/Cleaned/scan6")
NUM_IMAGES = 8

all_images = sorted(str(p) for p in DTU_SCAN.iterdir() if p.is_file())
# We seed before this call so the selection is reproducible; we also persist
# the chosen paths to metadata so the evaluator does not need to re-seed.
selected_images = sorted(random.sample(all_images, NUM_IMAGES))
print(f"Selected {len(selected_images)} images from {DTU_SCAN}")
for p in selected_images:
    print(" ", p)

images = load_and_preprocess_images(selected_images).to(device).float()
# load_and_preprocess_images returns [S, C, H, W]; VGGT expects [B, S, C, H, W].
images = images.unsqueeze(0)
B, S, C, H, W = images.shape
print("images:", images.shape, "range:", float(images.min()), float(images.max()))

In [ ]:
# Clean forward pass — also used as the reference target inside the attack loss.
with torch.no_grad():
    clean = model(images)

D_clean = clean["depth"].float()              # [B, S, H, W, 1]
C_clean = clean["depth_conf"].float()         # [B, S, H, W]
print("depth:", tuple(D_clean.shape), "mean:", float(D_clean.mean()),
      " depth_conf:", tuple(C_clean.shape), "mean:", float(C_clean.mean()))

In [ ]:
# PGD configuration. L∞ budget of 4/255 is the standard "barely visible" budget
# used in image-classifier adversarial robustness literature.
EPS = 4.0 / 255.0
STEP = 1.0 / 255.0
STEPS = 40
RANDOM_INIT = True       # random start inside the L∞ ball strengthens PGD
SHARE_DELTA_ACROSS_VIEWS = True  # one perturbation broadcast to all S views

# Why share across views? A single physical perturbation (e.g., a print on
# the scene or a sensor-side noise pattern) that fools every camera frame is
# the realistic transferability threat for multi-view models. The old notebook
# also shared the delta but didn't say why.

In [ ]:
def project_delta(delta_tensor: torch.Tensor, images_tensor: torch.Tensor, eps: float) -> torch.Tensor:
    """Project delta onto the intersection of:
        (a) the L∞ ball of radius eps, and
        (b) the box that keeps `images + delta` inside [0, 1] for every view.

    If delta is shared across views (shape [B, 1, C, H, W]), we reduce the per-view
    box constraint across the S axis so the projection still respects all views.
    """
    shared = delta_tensor.shape[1] == 1 and images_tensor.shape[1] > 1
    if shared:
        lo_pix = (-images_tensor).amax(dim=1, keepdim=True)     # = -min_s images
        hi_pix = (1.0 - images_tensor).amin(dim=1, keepdim=True)  # = 1 - max_s images
    else:
        lo_pix = -images_tensor
        hi_pix = 1.0 - images_tensor
    lo = torch.maximum(lo_pix, torch.full_like(lo_pix, -eps))
    hi = torch.minimum(hi_pix, torch.full_like(hi_pix, eps))
    return torch.clamp(delta_tensor, lo, hi)

In [ ]:
def si_log_depth_loss(D_adv: torch.Tensor,
                     D_clean: torch.Tensor,
                     conf_clean: torch.Tensor,
                     lam: float = 1.0,
                     eps_d: float = 1e-6) -> torch.Tensor:
    """Scale-invariant log-depth divergence, weighted by clean depth confidence.

    Loss = E[w * d^2] - lam * (E[w * d])^2,    where d = log D_adv - log D_clean.

    With lam=1 this is the *variance* of d — fully scale-invariant. Uniform
    rescale of depth gives zero loss, so the attacker is forced to break
    *relative* geometry rather than fall into the trivial "shift every depth
    by a constant factor" direction that the paper's plain L1 form gets
    absorbed by under ICP-with-scale alignment. Eigen et al. used lam=0.5
    for monocular depth *prediction* because they wanted the network to
    learn absolute scale too; for an attack we want the opposite.

    Confidence weights are .detach()'d so they act as fixed importance
    weights on the clean predictions — the attacker cannot lower them
    by changing the input. This is the same "frozen Σ" trick the
    point-map attack uses for the same reason.
    """
    log_adv = torch.log(D_adv.clamp(min=eps_d))
    log_clean = torch.log(D_clean.clamp(min=eps_d))
    d = log_adv - log_clean                      # [B, S, H, W, 1]
    w = conf_clean.detach().unsqueeze(-1)        # [B, S, H, W, 1]
    w = w / w.mean().clamp(min=eps_d)            # normalize so loss scale is comparable
    sq = (w * d * d).mean()
    mu = (w * d).mean()
    return sq - lam * mu * mu

In [ ]:
delta_shape = (B, 1, C, H, W) if SHARE_DELTA_ACROSS_VIEWS else (B, S, C, H, W)
# Construct a single leaf tensor and write the initial value into its .data,
# so requires_grad_ is set exactly once on the leaf and the initial projection
# does not accidentally produce a non-leaf via clamp().
delta = torch.zeros(delta_shape, device=device, dtype=torch.float32)
if RANDOM_INIT:
    delta.data.uniform_(-EPS, EPS)
delta.data.copy_(project_delta(delta.data, images, EPS))
delta.requires_grad_(True)

In [ ]:
torch.cuda.empty_cache()
loss_history = []

for i in range(STEPS):
    if delta.grad is not None:
        delta.grad.detach_()
        delta.grad.zero_()

    # `delta` broadcasts [B,1,C,H,W] -> [B,S,C,H,W] when shared.
    # The projection guarantees `images + delta` is already in [0,1], so we
    # skip the redundant .clamp() the old notebook had — .clamp kills gradient
    # at saturated pixels for no reason.
    x_adv = images + delta
    preds = model(x_adv)
    D_adv = preds["depth"].float()

    loss = si_log_depth_loss(D_adv, D_clean, C_clean, lam=0.5)

    (g,) = torch.autograd.grad(loss, delta)

    with torch.no_grad():
        # Sign step (untargeted ascent on the loss).
        delta.add_(STEP * g.sign())
        delta.copy_(project_delta(delta, images, EPS))

    loss_history.append(float(loss.detach()))
    if i % 5 == 0 or i == STEPS - 1:
        with torch.no_grad():
            linf = float(delta.abs().max())
        print(f"step {i:3d}  loss={loss_history[-1]:.6f}  ||delta||_inf={linf:.5f}")

adv_images_t = (images + delta).detach()
# Defensive: confirm the constraints survived the loop.
assert float(adv_images_t.min()) >= -1e-6
assert float(adv_images_t.max()) <= 1.0 + 1e-6
assert float((adv_images_t - images).abs().max()) <= EPS + 1e-6

In [ ]:
with torch.no_grad():
    advp = model(adv_images_t)

D_adv = advp["depth"].float()
C_adv = advp["depth_conf"].float()

# Scene-mean stats.
abs_err = (D_adv - D_clean).abs()
rel_err = abs_err / D_clean.clamp(min=1e-6)
log_err = torch.log(D_adv.clamp(min=1e-6)) - torch.log(D_clean.clamp(min=1e-6))

# Per-view SI-log (the variance of d per view), so the eval script can see
# which views were hit hardest.
per_view_d = log_err.squeeze(-1)        # [B, S, H, W]
per_view_var = per_view_d.var(dim=(-1, -2))            # [B, S]
per_view_mean = per_view_d.mean(dim=(-1, -2))          # [B, S]
per_view_si_log = per_view_var                          # variance of log-difference per view

# Per-pixel SI-log error map: (d - mean_d_view)^2, the integrand of the loss.
per_pixel_si_log = (per_view_d - per_view_mean.unsqueeze(-1).unsqueeze(-1)) ** 2  # [B, S, H, W]

summary = {
    "mean_depth_clean":     float(D_clean.mean()),
    "mean_depth_adv":       float(D_adv.mean()),
    "mean_abs_depth_err":   float(abs_err.mean()),
    "mean_rel_depth_err":   float(rel_err.mean()),
    "mean_abs_log_err":     float(log_err.abs().mean()),
    "si_log_loss_final":    float(per_pixel_si_log.mean()),  # unweighted, for cross-run comparison
    "mean_conf_clean":      float(C_clean.mean()),
    "mean_conf_adv":        float(C_adv.mean()),
    "linf_delta":           float(delta.detach().abs().max()),
    "l2_delta_per_pixel":   float(delta.detach().pow(2).mean().sqrt()),
    "per_view_si_log":      per_view_si_log.squeeze(0).tolist(),
}
print(json.dumps(summary, indent=2))

In [ ]:
out_dir = Path("output")
out_dir.mkdir(exist_ok=True)
out_clean_dir = out_dir / "clean_depth";  out_clean_dir.mkdir(exist_ok=True)
out_adv_dir   = out_dir / "adv_depth";    out_adv_dir.mkdir(exist_ok=True)
img_dir = Path("images/adv_depth");        img_dir.mkdir(parents=True, exist_ok=True)

def _np(t):
    return t.detach().cpu().numpy()

def _pose_enc_list(pred):
    return np.stack([_np(p) for p in pred["pose_enc_list"]], axis=0)

# Decode pose_enc -> extrinsics/intrinsics so the evaluator doesn't have to
# touch the encoding.
image_size_hw = (H, W)
clean_extr, clean_intr = pose_encoding_to_extri_intri(clean["pose_enc"].float(), image_size_hw)
adv_extr,   adv_intr   = pose_encoding_to_extri_intri(advp["pose_enc"].float(),  image_size_hw)

np.savez_compressed(
    out_clean_dir / "clean_depth.npz",
    depth=_np(D_clean),
    depth_conf=_np(C_clean),
    world_pts=_np(clean["world_points"]),
    world_pts_conf=_np(clean["world_points_conf"]),
    pose_enc=_np(clean["pose_enc"]),
    pose_enc_list=_pose_enc_list(clean),
    extrinsics=_np(clean_extr),
    intrinsics=_np(clean_intr),
)

np.savez_compressed(
    out_adv_dir / "adv_depth.npz",
    depth=_np(D_adv),
    depth_conf=_np(C_adv),
    world_pts=_np(advp["world_points"]),
    world_pts_conf=_np(advp["world_points_conf"]),
    pose_enc=_np(advp["pose_enc"]),
    pose_enc_list=_pose_enc_list(advp),
    extrinsics=_np(adv_extr),
    intrinsics=_np(adv_intr),
    # Extras the old notebook didn't save — these let the eval script
    # reproduce the attack and audit the budget.
    delta=_np(delta),
    adv_images=_np(adv_images_t),
    clean_images=_np(images),
    loss_history=np.array(loss_history, dtype=np.float32),
    per_view_si_log=_np(per_view_si_log),
    si_log_error_map=_np(per_pixel_si_log),
)

# Adversarial views as PNGs for visual inspection.
adv_S = adv_images_t.squeeze(0)
for s in range(adv_S.shape[0]):
    vutils.save_image(adv_S[s], img_dir / f"adv_s{s}.png", normalize=False)

# Environment block — cuDNN determinism is hardware-conditional, so the GPU
# and library versions matter for reproducibility.
env = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

metadata = {
    "seed": SEED,
    "model": "facebook/VGGT-1B",
    "scene": str(DTU_SCAN),
    "selected_images": selected_images,
    "image_shape": list(images.shape),
    "attack": {
        "type": "PGD (sign step, L_inf ball)",
        "loss": "scale-invariant log-depth (lam=1.0), confidence-weighted by clean depth_conf",
        "eps": EPS,
        "step": STEP,
        "steps": STEPS,
        "random_init": RANDOM_INIT,
        "share_delta_across_views": SHARE_DELTA_ACROSS_VIEWS,
    },
    "summary": summary,
    "env": env,
}
with open(out_dir / "attack_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("wrote:")
print(" ", out_clean_dir / "clean_depth.npz")
print(" ", out_adv_dir / "adv_depth.npz")
print(" ", out_dir / "attack_metadata.json")
print(" ", img_dir, "(PNG per view)")